💡Scenario: You work for a retail company that wants to monitor sales data in real-time. You have a streaming dataset representing online sales transactions. Your task is to calculate the total revenue generated per product category every minute and display it as a real-time dashboard.

Sample Dataset :

Dataset: You have a streaming dataset in CSV format with the following columns:
timestamp: The timestamp when the sale occurred.
product_id: The unique identifier of the product.
product_name: The name of the product.
category: The category to which the product belongs.
price: The price at which the product was sold.

Query: Write a PySpark Structured Streaming query that calculates the total revenue generated per product category every minute and updates it in real-time. You can assume that you have already set up a streaming source with the dataset.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import window, col, sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
from datetime import datetime

# Create a Spark session
spark = SparkSession.builder \
    .appName("RealTimeSalesDashboard") \
    .getOrCreate()

# Define schema for the dataset
schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", IntegerType(), True)
])

# Sample data with proper datetime objects
data = [
    (datetime.strptime("2024-12-18 10:00:00", "%Y-%m-%d %H:%M:%S"), 101, "Laptop", "Electronics", 1200),
    (datetime.strptime("2024-12-18 10:00:30", "%Y-%m-%d %H:%M:%S"), 102, "Phone", "Electronics", 800),
    (datetime.strptime("2024-12-18 10:01:10", "%Y-%m-%d %H:%M:%S"), 103, "Shoes", "Apparel", 150),
    (datetime.strptime("2024-12-18 10:01:45", "%Y-%m-%d %H:%M:%S"), 104, "Watch", "Accessories", 200),
    (datetime.strptime("2024-12-18 10:02:05", "%Y-%m-%d %H:%M:%S"), 105, "Tablet", "Electronics", 400),
    (datetime.strptime("2024-12-18 10:02:30", "%Y-%m-%d %H:%M:%S"), 106, "Shirt", "Apparel", 50)
]

# Create a static DataFrame
df = spark.createDataFrame(data, schema=schema)

# Simulate streaming by treating it as a batch process
category_revenue = df \
    .groupBy(window(col("timestamp"), "1 minute"), col("category")) \
    .agg(sum(col("price")).alias("total_revenue")) \
    .orderBy("window")

# Show the results
category_revenue.show(truncate=False)


+------------------------------------------+-----------+-------------+
|window                                    |category   |total_revenue|
+------------------------------------------+-----------+-------------+
|{2024-12-18 10:00:00, 2024-12-18 10:01:00}|Electronics|2000         |
|{2024-12-18 10:01:00, 2024-12-18 10:02:00}|Apparel    |150          |
|{2024-12-18 10:01:00, 2024-12-18 10:02:00}|Accessories|200          |
|{2024-12-18 10:02:00, 2024-12-18 10:03:00}|Electronics|400          |
|{2024-12-18 10:02:00, 2024-12-18 10:03:00}|Apparel    |50           |
+------------------------------------------+-----------+-------------+

